In [1]:
# Cellule 1 — Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Cellule 2 — Code du dépôt
!rm -rf /content/repo
!git clone -q https://github.com/nerminecheriaa/beam-prediction-deepsense6g.git /content/repo
import sys; sys.path.insert(0, "/content/repo")
!cd /content/repo && git log --oneline -1

3913e72 (HEAD -> main, origin/main, origin/HEAD) A2 : to_model_input avec option augment


In [3]:
# Cellule 3 — Chemins + cache images
import os, torch, pandas as pd
from src.dataset import build_image_cache, get_split
from src.models import CameraResNet18
from src.train import train_model, predict, set_seed
from src.evaluate import topk_accuracy

ROOT = "/content/data"
SPLITS = "/content/repo/splits.csv"
DRIVE_OUT = "/content/drive/MyDrive/beam_data/camera"
CACHE = f"{DRIVE_OUT}/cam_cache_224.pt"
device = "cuda"
print(torch.cuda.get_device_name(0))

cache = build_image_cache(SPLITS, ROOT, CACHE)   # doit afficher « Cache rechargé »

Tesla T4
Cache rechargé depuis /content/drive/MyDrive/beam_data/camera/cam_cache_224.pt : (3235, 3, 224, 224)


In [4]:
# Cellule 4 — Préparation A1
SEEDS = [0, 1, 2, 3, 4]                          # mêmes graines que Mériem
RUN_DIR = f"{DRIVE_OUT}/seeds_baseline"
os.makedirs(RUN_DIR, exist_ok=True)

x_tr, y_tr, idx_tr = get_split(cache, "train")
x_val, y_val, idx_val = get_split(cache, "val")
x_te, y_te, idx_te = get_split(cache, "test")
splits = {"train": (x_tr, y_tr, idx_tr), "val": (x_val, y_val, idx_val), "test": (x_te, y_te, idx_te)}
print({k: len(v[1]) for k, v in splits.items()})   # attendu : 2265 / 490 / 480

{'train': 2265, 'val': 490, 'test': 480}


In [7]:
# Cellule 5 — Entraînement sur 5 graines (reprise automatique)
rows = []
for s in SEEDS:
    out_path = f"{RUN_DIR}/camera_outputs_seed{s}.pt"
    if os.path.exists(out_path):                              # graine déjà faite : on recharge
        out = torch.load(out_path, weights_only=False)
        print(f"graine {s} : déjà faite, rechargée")
    else:
        set_seed(s)                                           # AVANT le modèle : la tête 64 classes dépend aussi de la graine
        model = CameraResNet18().to(device)
        ckpt = f"{RUN_DIR}/cam_seed{s}.pth"
        train_model(model, x_tr, y_tr, x_val, y_val, device, ckpt, cfg={"seed": s})

        ck = torch.load(ckpt, weights_only=False)             # meilleure époque, choisie sur val
        model.load_state_dict(ck["model_state"])
        out = {"seed": s, "best_epoch": ck["epoch"]}
        for name, (x, y, idx) in splits.items():
            lo, fe = predict(model, x, device, return_features=True)
            out[name] = {"logits": lo, "features": fe, "labels": y, "index": idx}
        torch.save(out, out_path)

    va = topk_accuracy(out["val"]["logits"], y_val)
    te = topk_accuracy(out["test"]["logits"], y_te)
    rows.append({"seed": s, "best_epoch": out["best_epoch"],
                 **{f"val_{k}": 100*v for k, v in va.items()},
                 **{f"test_{k}": 100*v for k, v in te.items()}})
    print(f"=== graine {s} :", {k: round(v, 1) for k, v in rows[-1].items()})

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 199MB/s]


ep  1 | train loss 3.387 top1 0.210 | val loss 4.104 top1 0.051 top3 0.147 top5 0.237 | 8.1s  <- meilleur (sauvegardé)
ep  2 | train loss 2.257 top1 0.396 | val loss 3.536 top1 0.120 top3 0.284 top5 0.369 | 3.5s  <- meilleur (sauvegardé)
ep  3 | train loss 1.724 top1 0.490 | val loss 3.344 top1 0.137 top3 0.335 top5 0.433 | 3.5s  <- meilleur (sauvegardé)
ep  4 | train loss 1.419 top1 0.576 | val loss 3.037 top1 0.180 top3 0.422 top5 0.569 | 3.5s  <- meilleur (sauvegardé)
ep  5 | train loss 1.195 top1 0.629 | val loss 3.094 top1 0.161 top3 0.369 top5 0.504 | 3.5s
ep  6 | train loss 0.995 top1 0.710 | val loss 2.966 top1 0.251 top3 0.459 top5 0.576 | 3.5s  <- meilleur (sauvegardé)
ep  7 | train loss 0.827 top1 0.765 | val loss 3.127 top1 0.161 top3 0.365 top5 0.498 | 3.5s
ep  8 | train loss 0.681 top1 0.816 | val loss 3.084 top1 0.180 top3 0.386 top5 0.524 | 3.7s
ep  9 | train loss 0.530 top1 0.872 | val loss 3.179 top1 0.167 top3 0.343 top5 0.471 | 3.6s
ep 10 | train loss 0.413 top1 0.9

In [8]:
# Cellule 6 — Résumé A1 (moyenne ± écart-type)
res = pd.DataFrame(rows)
res.to_csv(f"{RUN_DIR}/summary_baseline.csv", index=False)
print(res.round(1).to_string(index=False))
print("\nMoyenne ± écart-type :")
print(res.drop(columns="seed").agg(["mean", "std"]).round(1).to_string())
print("\nFichiers :", sorted(os.listdir(RUN_DIR)))

 seed  best_epoch  val_top1  val_top3  val_top5  test_top1  test_top3  test_top5
    0           6      25.1      45.9      57.6       20.2       37.9       49.8
    1           6      29.2      54.9      65.3       19.0       40.0       50.8
    2          10      26.9      50.4      58.6       16.0       30.0       41.7
    3           6      25.5      48.6      59.0       14.2       31.0       43.3
    4           4      23.5      48.8      62.4       11.0       30.2       43.8

Moyenne ± écart-type :
      best_epoch  val_top1  val_top3  val_top5  test_top1  test_top3  test_top5
mean         6.4      26.0      49.7      60.6       16.1       33.8       45.9
std          2.2       2.1       3.3       3.2        3.7        4.8        4.1

Fichiers : ['cam_seed0.pth', 'cam_seed0_history.json', 'cam_seed1.pth', 'cam_seed1_history.json', 'cam_seed2.pth', 'cam_seed2_history.json', 'cam_seed3.pth', 'cam_seed3_history.json', 'cam_seed4.pth', 'cam_seed4_history.json', 'camera_outputs_seed0.

In [5]:
# Cellule 7 — Test rapide A2 (augmentation + étiquettes douces)
from src.dataset import to_model_input
from src.train import soft_targets
xa = to_model_input(x_tr[:8], device, augment=True)
xn = to_model_input(x_tr[:8], device)
print("forme :", tuple(xa.shape), "| part des pixels modifiés :", round((xa != xn).float().mean().item(), 2))
t = soft_targets(torch.tensor([16]), 64, 1.0)
print("cible douce autour du faisceau 16 :", t[0, 14:19].round(decimals=3).tolist(), "| somme =", round(t.sum().item(), 3))

forme : (8, 3, 224, 224) | part des pixels modifiés : 0.99
cible douce autour du faisceau 16 : [0.05400000140070915, 0.24199999868869781, 0.39899998903274536, 0.24199999868869781, 0.05400000140070915] | somme = 1.0


In [6]:
# Cellule 8 — A2 : 3 variantes x 5 graines (on n'affiche QUE la validation)
import io, contextlib
VARIANTS = {
    "aug":      {"augment": True,  "soft_sigma": 0.0},
    "soft":     {"augment": False, "soft_sigma": 1.0},
    "aug_soft": {"augment": True,  "soft_sigma": 1.0},
}
VAR_ROOT = f"{DRIVE_OUT}/variants"

rows_var = []
for name, extra in VARIANTS.items():
    vdir = f"{VAR_ROOT}/{name}"; os.makedirs(vdir, exist_ok=True)
    for s in SEEDS:
        out_path = f"{vdir}/camera_outputs_seed{s}.pt"
        if os.path.exists(out_path):
            out = torch.load(out_path, weights_only=False)
        else:
            set_seed(s)
            model = CameraResNet18().to(device)
            ckpt = f"{vdir}/cam_seed{s}.pth"
            with contextlib.redirect_stdout(io.StringIO()):          # masque les 20 lignes d'époques
                train_model(model, x_tr, y_tr, x_val, y_val, device, ckpt, cfg={"seed": s, **extra})
            ck = torch.load(ckpt, weights_only=False)
            model.load_state_dict(ck["model_state"])
            out = {"seed": s, "best_epoch": ck["epoch"], "variant": name, "cfg": extra}
            for sp, (x, y, idx) in splits.items():                   # test calculé et sauvegardé, PAS affiché
                lo, fe = predict(model, x, device, return_features=True)
                out[sp] = {"logits": lo, "features": fe, "labels": y, "index": idx}
            torch.save(out, out_path)
        va = topk_accuracy(out["val"]["logits"], y_val)
        rows_var.append({"variant": name, "seed": s, "best_epoch": out["best_epoch"],
                         **{f"val_{k}": 100*v for k, v in va.items()}})
        print(f"{name:9s} graine {s} | époque {out['best_epoch']:2d} | val",
              {k: round(100*v, 1) for k, v in va.items()})

aug       graine 0 | époque  5 | val {'top1': 28.0, 'top3': 54.7, 'top5': 69.0}
aug       graine 1 | époque  6 | val {'top1': 33.3, 'top3': 63.5, 'top5': 75.3}
aug       graine 2 | époque 10 | val {'top1': 32.7, 'top3': 59.8, 'top5': 69.6}
aug       graine 3 | époque  8 | val {'top1': 30.6, 'top3': 55.1, 'top5': 66.3}
aug       graine 4 | époque  6 | val {'top1': 31.4, 'top3': 53.9, 'top5': 67.6}
soft      graine 0 | époque  5 | val {'top1': 20.0, 'top3': 49.8, 'top5': 64.3}
soft      graine 1 | époque 10 | val {'top1': 26.9, 'top3': 51.2, 'top5': 65.5}
soft      graine 2 | époque 10 | val {'top1': 23.7, 'top3': 55.5, 'top5': 70.6}
soft      graine 3 | époque  9 | val {'top1': 28.6, 'top3': 61.2, 'top5': 72.9}
soft      graine 4 | époque  4 | val {'top1': 20.2, 'top3': 48.6, 'top5': 66.5}
aug_soft  graine 0 | époque 14 | val {'top1': 25.5, 'top3': 53.5, 'top5': 69.4}
aug_soft  graine 1 | époque 16 | val {'top1': 31.4, 'top3': 59.0, 'top5': 71.0}
aug_soft  graine 2 | époque 18 | val {'t

In [7]:
# Cellule 9 — Comparaison VAL : A1 vs variantes, et décision
base = pd.read_csv(f"{DRIVE_OUT}/seeds_baseline/summary_baseline.csv")
base = base[["seed", "best_epoch", "val_top1", "val_top3", "val_top5"]].assign(variant="A1_baseline")
allv = pd.concat([base, pd.DataFrame(rows_var)], ignore_index=True)
allv["val_sum"] = allv.val_top1 + allv.val_top3 + allv.val_top5
allv.to_csv(f"{VAR_ROOT}/summary_variants_val.csv", index=False)

order = ["A1_baseline", "aug", "soft", "aug_soft"]
g = allv.groupby("variant")[["val_top1", "val_top3", "val_top5", "val_sum", "best_epoch"]]
table = g.mean().round(1).astype(str) + " ± " + g.std().round(1).astype(str)
print(table.loc[order].to_string())

means = g.mean()["val_sum"]
best = means.loc[order].idxmax()
print(f"\nMeilleure moyenne val (Top-1+3+5) : {best} = {means[best]:.1f}  (A1 = {means['A1_baseline']:.1f})")
print("DÉCISION :", "on garde A1" if best == "A1_baseline" else f"on adopte '{best}'")

               val_top1    val_top3    val_top5       val_sum  best_epoch
variant                                                                  
A1_baseline  26.0 ± 2.1  49.7 ± 3.3  60.6 ± 3.2   136.3 ± 7.8   6.4 ± 2.2
aug          31.2 ± 2.1  57.4 ± 4.1  69.6 ± 3.5   158.1 ± 8.9   7.0 ± 2.0
soft         23.9 ± 3.9  53.3 ± 5.2  68.0 ± 3.6  145.1 ± 11.7   7.6 ± 2.9
aug_soft     28.3 ± 2.7  57.9 ± 3.7  72.6 ± 3.0   158.8 ± 8.6  12.0 ± 5.7

Meilleure moyenne val (Top-1+3+5) : aug_soft = 158.8  (A1 = 136.3)
DÉCISION : on adopte 'aug_soft'


In [8]:
# Cellule 10 — A3 : test UNE SEULE FOIS pour la variante retenue + export seeds_final
import shutil
CHOSEN = "aug_soft"                                   # décidé sur la validation (cellule 9)
VAR_ROOT = f"{DRIVE_OUT}/variants"
SRC_DIR = f"{VAR_ROOT}/{CHOSEN}"
FINAL = f"{DRIVE_OUT}/seeds_final"
os.makedirs(FINAL, exist_ok=True)

rows_f = []
for s in SEEDS:
    p = f"{SRC_DIR}/camera_outputs_seed{s}.pt"
    out = torch.load(p, weights_only=False)
    shutil.copy(p, f"{FINAL}/camera_outputs_seed{s}.pt")      # même format que seeds_baseline
    va = topk_accuracy(out["val"]["logits"], y_val)
    te = topk_accuracy(out["test"]["logits"], y_te)
    rows_f.append({"seed": s, "best_epoch": out["best_epoch"],
                   **{f"val_{k}": 100*v for k, v in va.items()},
                   **{f"test_{k}": 100*v for k, v in te.items()}})

fin = pd.DataFrame(rows_f)
fin.to_csv(f"{FINAL}/summary_final.csv", index=False)
print(f"Variante retenue : {CHOSEN}\n")
print(fin.round(1).to_string(index=False))

base = pd.read_csv(f"{DRIVE_OUT}/seeds_baseline/summary_baseline.csv")
cols = ["test_top1", "test_top3", "test_top5"]
fmt = lambda d: " / ".join(f"{d[c].mean():.1f} ± {d[c].std():.1f}" for c in cols)
print("\nTEST (moyenne ± écart-type, Top-1 / Top-3 / Top-5)")
print("  Fréquence (plancher) : 13.1 / 36.3 / 42.3")
print("  A1 (sans régul.)     :", fmt(base))
print(f"  {CHOSEN:21s}:", fmt(fin))
print("\nFichiers seeds_final :", sorted(os.listdir(FINAL)))

Variante retenue : aug_soft

 seed  best_epoch  val_top1  val_top3  val_top5  test_top1  test_top3  test_top5
    0          14      25.5      53.5      69.4       14.8       41.2       52.7
    1          16      31.4      59.0      71.0       15.6       37.7       49.8
    2          18      27.1      55.7      73.5       21.3       39.4       49.4
    3           6      26.3      58.0      71.8       14.2       35.0       46.5
    4           6      31.0      63.3      77.3       15.2       36.5       47.9

TEST (moyenne ± écart-type, Top-1 / Top-3 / Top-5)
  Fréquence (plancher) : 13.1 / 36.3 / 42.3
  A1 (sans régul.)     : 16.1 ± 3.7 / 33.8 ± 4.8 / 45.9 ± 4.1
  aug_soft             : 16.2 ± 2.9 / 38.0 ± 2.4 / 49.3 ± 2.3

Fichiers seeds_final : ['camera_outputs_seed0.pt', 'camera_outputs_seed1.pt', 'camera_outputs_seed2.pt', 'camera_outputs_seed3.pt', 'camera_outputs_seed4.pt', 'summary_final.csv']
